# Prediciendo la Fuga de Clientes (Customer Churn)

## **Objetivo**: Identificar clientes que están a punto de abandonar el servicio

Somos parte de un equipo de datos de una empresa de telecomunicaciones. Nuestra misión es crucial: predecir qué clientes tienen más probabilidades de abandonar nuestro servicio (hacer "churn").

Si pudiéramos identificar a estos clientes de antemano, podríamos ofrecerles descuentos o planes especiales para retenerlos. ¡Esto puede salvar a la empresa de perder mucho dinero!

Este es un problema de clasificación binaria, porque intentamos predecir una de dos posibles categorías: "Sí, abandonará" (Churn = Yes) o "No, no abandonará" (Churn = No).

## Paso 1: Configuración del Entorno e Importación de Librerías

Usaremos las mismas librerías que antes, pero añadiremos algunas herramientas específicas para la evaluación de modelos de clasificación.

- **sklearn.metrics**: Además de las métricas de regresión, ahora usaremos confusion_matrix y classification_report, que son fundamentales para la clasificación.
- **sklearn.ensemble**: Aquí encontraremos un modelo más potente llamado RandomForestClassifier.

In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de modelado y evaluación
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Configuramos el estilo de los gráficos
sns.set_style("whitegrid")

## Paso 2: Cargar y Conocer los Datos

Carguemos el dataset de clientes de la empresa de telecomunicaciones.

**Importante**: Asegúrate de que el archivo Telco-Customer-Churn.csv está en la misma carpeta que este notebook.

In [ ]:
# Cargamos el dataset
df = pd.read_csv('Telco-Customer-Churn.csv')

# Mostramos las primeras 5 filas
print("Primeras 5 filas del dataset:")
df.head()

**Información General del Dataset**

In [ ]:
# Usamos .info() para obtener un resumen de los datos
print("Información general del dataset:")
df.info()

*Observación Importante*: ¡Tenemos un problema! La columna TotalCharges debería ser un número (como MonthlyCharges), pero aparece como object (texto). Esto suele deberse a valores no numéricos o vacíos. ¡Vamos a arreglarlo!

## Paso 3: Limpieza y Preparación de Datos

Un paso fundamental en cualquier proyecto de ML es limpiar y preparar los datos para que el modelo pueda entenderlos.

**a. Arreglar la columna TotalCharges**

Vamos a convertir TotalCharges a un tipo numérico. Los valores que no se puedan convertir (como espacios en blanco) se convertirán en NaN (Not a Number).

In [ ]:
# Convertimos la columna 'TotalCharges' a numérico. Los errores se convertirán en NaN.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificamos si hay valores nulos (NaN) después de la conversión
print(f"¿Hay valores nulos en 'TotalCharges'?: {df['TotalCharges'].isnull().sum()}")

# Mostremos las filas que tienen TotalCharges nulo
print("Filas con TotalCharges nulo:")
df[df['TotalCharges'].isnull()]

*Conclusión*: Son solo 11 clientes. Todos tienen una tenure (antigüedad) de 0 meses, lo que significa que son clientes nuevos y aún no han acumulado cargos totales. Tiene sentido rellenar estos valores nulos con 0.

In [ ]:
# Rellenamos los valores nulos con 0
df['TotalCharges'].fillna(0, inplace=True)

# Verificamos que ya no hay nulos
print(f"¿Hay valores nulos después de rellenar?: {df['TotalCharges'].isnull().sum()}")

**b. Eliminar columnas innecesarias**

La columna customerID es un identificador único para cada cliente y no contiene información útil para predecir el churn. La eliminaremos.

In [ ]:
# Eliminamos la columna customerID
df.drop('customerID', axis=1, inplace=True)

print("Columna 'customerID' eliminada.")

## Paso 4: Exploración y Visualización de Datos (EDA)

Ahora que los datos están limpios, vamos a explorarlos para encontrar patrones que nos ayuden a entender por qué los clientes abandonan.

**Distribución de la Variable Objetivo (Churn)**

Lo primero es ver cuántos clientes abandonan y cuántos no.

In [ ]:
# Visualizamos la distribución de la variable objetivo 'Churn'
plt.figure(figsize=(8, 6))
sns.countplot(x='Churn', data=df)
plt.title('Distribución de Clientes que Abandonan (Churn)')
plt.xlabel('¿Abandonó el servicio?')
plt.ylabel('Número de Clientes')
plt.show()

# También podemos ver los porcentajes
churn_percentage = df['Churn'].value_counts(normalize=True) * 100
print(f"Porcentaje de clientes que NO abandonan: {churn_percentage['No']:.2f}%")
print(f"Porcentaje de clientes que SÍ abandonan: {churn_percentage['Yes']:.2f}%")

*Conclusión*: El dataset está desbalanceado. Tenemos muchos más clientes que no abandonan (73%) que los que sí lo hacen (26%). Es importante tenerlo en cuenta al evaluar nuestro modelo.

**¿El tipo de contrato influye en el churn?**

In [ ]:
# ¿El tipo de contrato influye en el churn?
plt.figure(figsize=(10, 6))
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Churn según el Tipo de Contrato')
plt.xlabel('Tipo de Contrato')
plt.ylabel('Número de Clientes')
plt.show()

*Conclusión*: ¡Claramente sí! Los clientes con contrato mes a mes tienen una tasa de churn muchísimo más alta. Esta será una variable muy predictiva.

**¿Cómo influyen los cargos mensuales y la antigüedad?**

In [ ]:
# Creemos dos gráficos, uno para MonthlyCharges y otro para tenure
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de MonthlyCharges
sns.histplot(data=df, x='MonthlyCharges', hue='Churn', multiple='stack', kde=True, ax=axes[0])
axes[0].set_title('Distribución de Cargos Mensuales por Churn')
axes[0].set_xlabel('Cargos Mensuales ($)')

# Gráfico de tenure (antigüedad en meses)
sns.histplot(data=df, x='tenure', hue='Churn', multiple='stack', kde=True, ax=axes[1])
axes[1].set_title('Distribución de Antigüedad (tenure) por Churn')
axes[1].set_xlabel('Antigüedad (meses)')

plt.show()

*Conclusiones*:

- **Cargos Mensuales**: Los clientes con cargos mensuales más altos parecen abandonar más.
- **Antigüedad**: La mayoría de los clientes que abandonan lo hacen al principio de su relación con la empresa (los primeros meses). ¡Esto tiene mucho sentido!

## Paso 5: Preparación de los Datos para el Modelo

Al igual que antes, necesitamos convertir todas nuestras variables a un formato numérico.

**a. Separar Características (X) y Objetivo (y)**

In [ ]:
# Nuestro objetivo (y) es la columna 'Churn'
y = df['Churn']

# Nuestras características (X) son todas las demás columnas
X = df.drop('Churn', axis=1)

print("Forma de las características (X):", X.shape)
print("Forma del objetivo (y):", y.shape)

**b. Convertir Variables Categóricas a Numéricas (One-Hot Encoding)**

Tenemos muchas columnas de texto (gender, Partner, PhoneService, etc.). Las convertiremos todas a la vez usando pd.get_dummies.

In [ ]:
# Identificamos las columnas categóricas
categorical_cols = X.select_dtypes(include=['object']).columns

# Aplicamos One-Hot Encoding a todas las columnas categóricas
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print("Nuestras características (X) después del One-Hot Encoding:")
print(X.head())

*Explicación*: Ahora tenemos una columna para cada categoría. Por ejemplo, en lugar de Contract_Month-to-month, tenemos Contract_One year y Contract_Two year. Si ambas son 0, significa que el contrato es mes a mes.

## Paso 6: Dividir los Datos en Entrenamiento y Prueba

El proceso es idéntico a los ejercicios anteriores.

In [ ]:
# Dividimos los datos en 80% para entrenamiento y 20% para prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Usamos 'stratify=y' para mantener la misma proporción de Churn/No-Churn en ambos conjuntos.
# Esto es muy importante en datasets desbalanceados.

print(f"Tamaño del conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Tamaño del conjunto de prueba: {X_test.shape[0]} muestras")

## Paso 7: Entrenar un Modelo de Clasificación

Empezaremos con un modelo simple y potente: Random Forest Classifier. Es un conjunto de muchos árboles de decisión que trabajan juntos para dar una predicción muy robusta.

In [ ]:
# 1. Creamos una instancia del modelo Random Forest
# n_estimators es el número de árboles en el bosque.
rf_model = LogisticRegression()

# 2. Entrenamos el modelo con los datos de entrenamiento
print("Entrenando el modelo Logistic Regression...")
rf_model.fit(X_train, y_train)
print("¡Modelo entrenado con éxito!")

## Paso 8: Evaluar el Modelo de Clasificación

Aquí es donde la evaluación de clasificación se vuelve interesante. No basta con decir "acertó o no".

**a. Hacer Predicciones**

In [ ]:
# Hacemos predicciones sobre el conjunto de prueba
y_pred = rf_model.predict(X_test)

**b. La Matriz de Confusión**

Es la herramienta más importante para entender los errores de nuestro modelo.

In [ ]:
# Calculamos la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

# La visualizamos con un mapa de calor (heatmap)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn', 'Churn'], 
            yticklabels=['No Churn', 'Churn'])
plt.title('Matriz de Confusión')
plt.ylabel('Valor Real')
plt.xlabel('Valor Predicho')
plt.show()

**¿Cómo leer la matriz?**

- Esquina Superior Izquierda: Verdaderos Negativos. El modelo predijo "No Churn" y era correcto.
- Esquina Inferior Derecha: Verdaderos Positivos. El modelo predijo "Churn" y era correcto.
- Esquina Superior Derecha: Falsos Positivos. El modelo predijo "Churn", pero el cliente no abandonó. (Error menos grave).
- Esquina Inferior Izquierda: Falsos Negativos. El modelo predijo "No Churn", pero el cliente SÍ abandonó. **¡Este es el error más costoso para la empresa!**

**c. Reporte de Clasificación**

Este reporte nos da las métricas clave: Precisión, Recall (Sensibilidad) y F1-Score.

In [ ]:
# Generamos el reporte de clasificación
report = classification_report(y_test, y_pred)
print("Reporte de Clasificación:")
print(report)

**Interpretación de las métricas (para la clase "Yes" - Churn):**

- Precision: De todos los clientes que el modelo dijo que abandonarían, el XX% realmente lo hicieron.
- Recall: De todos los clientes que realmente abandonaron, el modelo solo identificó correctamente al XX%. ¡Nuestro modelo se está perdiendo al XX% de los clientes que se van! (Esto corresponde a los Falsos Negativos).
- F1-Score: Es una media armónica entre precisión y recall. Nos da una única métrica de rendimiento para la clase "Churn".
- Accuracy: El modelo acierta el 79% de las veces. Pero, ¡cuidado! Esta métrica es engañosa en datasets desbalanceados. Si un modelo siempre predijera "No Churn", tendría una accuracy del 73%, ¡sin haber aprendido nada!

## Paso 9: ¿Qué factores son más importantes para predecir el Churn?

Una gran ventaja del Random Forest es que puede decirnos qué características fueron las más influyentes para hacer sus predicciones.

In [ ]:
# Obtenemos la importancia de las características del modelo
importances = rf_model.feature_importances_
features = X.columns
indices = np.argsort(importances)

# Visualizamos las 10 características más importantes
plt.figure(figsize=(12, 8))
plt.title('Top 10 Características Más Importantes')
plt.barh(range(10), importances[indices[-10:]], color='b', align='center')
plt.yticks(range(10), [features[i] for i in indices[-10:]])
plt.xlabel('Importancia Relativa')
plt.show()

*Conclusión*: ¡Aquí tenemos información de oro para el negocio! Los factores más determinantes para el churn.

**¿Qué hemos aprendido?**

- A limpiar datos y manejar problemas comunes como tipos de datos incorrectos.
- A explorar datos para encontrar patrones de negocio.
- A entrenar un modelo de clasificación (Random Forest).
- A evaluar el modelo con herramientas clave como la matriz de confusión y el reporte de clasificación, entendiendo la diferencia entre precisión y recall.
- A interpretar el modelo para extraer insights accionables para el negocio.

**¿Cómo podríamos mejorarlo?**

- **Balancear las clases**: Podríamos usar técnicas como SMOTE para crear más ejemplos de la clase minoritaria ("Churn") y ayudar al modelo a aprender mejor sobre ella.
- **Optimizar el modelo**: Podríamos ajustar los "hiperparámetros" del Random Forest (como n_estimators) para encontrar una combinación que funcione mejor.
- **Probar otros modelos**: Algoritmos como XGBoost o LightGBM a menudo ofrecen un rendimiento aún mejor en este tipo de problemas.